# Test standardized personas

In [1]:
%load_ext autoreload
%autoreload 2

import importlib
import sys
from pathlib import Path

import pandas as pd

CODE_DIR = Path("../..").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from data import persona_loaders, persona_spec
importlib.reload(persona_spec)
importlib.reload(persona_loaders)

specs = persona_spec.load_persona_spec()
combined = persona_loaders.load_all_personas(specs=specs)

print(f"Loaded {len(combined):,} participants | active attributes: {list(specs)}")
for persona in specs.keys():
    print(persona)
    print(combined[persona].unique())
    print('+'*50)

combined.head()

Loaded 72,402 participants | active attributes: ['country', 'gender', 'age', 'region', 'state', 'education', 'family_income', 'income', 'race', 'hispanic', 'religion', 'religious_attendance', 'political_party', 'political_ideology', 'marital_status', 'employment_status', 'household_size', 'children_in_household', 'children_ever_had', 'urban_rural', 'voted_last_election', 'union_member', 'residence_type', 'citizen', 'employer_sector', 'spouse_education', 'spouse_employment', 'nutrition_label_use', 'ancestry', 'personal_income', 'occupation', 'town_size', 'immigrant_status', 'lives_with_parents', 'home_language', 'chief_wage_earner', 'family_savings', 'social_class', 'voting_frequency', 'sexual_orientation', 'diet_quality', 'medical_reading_help', 'bmi', 'snap_benefits', 'wic_benefits', 'online_grocery_share', 'nutrition_literacy_score', 'political_label', 'owns_home', 'rents_home', 'military_service', 'electric_heating', 'political_interest', 'electricity_included_in_rent', 'knows_elect

,dataset,CASEID,country,gender,age,region,state,education,family_income,income,...,social_media_shares_media,social_media_follows_accounts,social_media_multiple_accounts,social_media_perceived_usefulness,social_media_follows_friends_shares,social_media_checks_friends_activity,online_shopping_frequency,takeout_delivery_frequency,online_grocery_frequency,vote_choice_2024
0,issp,2015:2015008400000003,United States,Male,65+,New England,NaN,"Lower level tertiary, first stage","$75,000-$89,999",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,issp,2015:2015008400000005,United States,Female,55-64,New England,NaN,"Upper level tertiary (Master, Doctor)","$170,000 or more",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,issp,2015:2015008400000008,United States,Female,18-24,Middle Atlantic,NaN,Upper secondary education,"$30,000-$34,999",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,issp,2015:2015008400000010,United States,Male,65+,Middle Atlantic,NaN,Post-secondary non-tertiary education,"$60,000-$74,999",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,issp,2015:2015008400000011,United States,Female,25-34,Middle Atlantic,NaN,Upper secondary education,"Less than $1,000",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
combined['political_ideology'].value_counts(dropna=False)

political_ideology
NaN                          52814
Moderate                      4491
Liberal                       1918
Conservative                  1684
Very liberal                  1394
Somewhat conservative         1236
Very conservative             1235
Somewhat liberal              1103
5 (0 = Left, 10 = Right)       847
5 (1 = Left, 10 = Right)       615
3 (1 = Left, 10 = Right)       308
10 (0 = Left, 10 = Right)      268
6 (1 = Left, 10 = Right)       260
1 (1 = Left, 10 = Right)       248
7 (1 = Left, 10 = Right)       242
7 (0 = Left, 10 = Right)       239
8 (1 = Left, 10 = Right)       239
6 (0 = Left, 10 = Right)       206
4 (1 = Left, 10 = Right)       206
3 (0 = Left, 10 = Right)       204
4 (0 = Left, 10 = Right)       203
8 (0 = Left, 10 = Right)       182
10 (1 = Left, 10 = Right)      182
2 (0 = Left, 10 = Right)       164
0 (0 = Left, 10 = Right)       145
2 (1 = Left, 10 = Right)       135
Slightly liberal               130
Slightly conservative          127
9

In [2]:
# dataset_order = list(persona_loaders.RAW_LOADERS.keys())
# for attr in specs:
#     present_ds = [ds for ds in dataset_order if combined.loc[combined["dataset"] == ds, attr].notna().any()]
#     sub = combined.loc[combined["dataset"].isin(present_ds), ["dataset", attr]].copy()
#     sub[attr] = sub[attr].where(sub[attr].notna(), "<NA>").astype(str)
#     print(f"\n=== {attr} (non-null: {int((sub[attr] != '<NA>').sum()):,}, <NA>: {int((sub[attr] == '<NA>').sum()):,}, unique: {combined[attr].nunique()}) ===")
#     table = pd.crosstab(sub[attr], sub["dataset"], margins=True)[present_ds + ["All"]]
#     display(table)

In [2]:
def inspect_dataset_personas(ds_name: str) -> None:
    sub = combined[combined["dataset"] == ds_name]
    if sub.empty:
        raise ValueError(f"Unknown or empty dataset {ds_name!r}")
    active_attrs = [a for a in specs if a in sub.columns and sub[a].notna().any()]
    n = len(sub)
    print(f"=== {ds_name}: {n:,} participants | {len(active_attrs)} active persona attributes ===")
    for attr in active_attrs:
        n_present = int(sub[attr].notna().sum())
        n_missing = n - n_present
        n_unique = int(sub[attr].nunique(dropna=True))
        s = sub[attr].where(sub[attr].notna(), "<NA>").astype(str)
        vc = s.value_counts()
        tbl = pd.DataFrame({"count": vc, "pct": (vc / n * 100).round(2)})
        tbl.index.name = attr
        print(f"\n{attr} (present={n_present:,}, missing={n_missing:,}, unique={n_unique})")
        display(tbl)

# ISSP

In [4]:
inspect_dataset_personas("issp")

=== issp: 14,365 participants | 22 active persona attributes ===

country (present=14,365, missing=0, unique=1)


,count,pct
country,,
United States,14365,100.0



gender (present=14,335, missing=30, unique=2)


,count,pct
gender,,
Female,7906,55.04
Male,6429,44.75
<NA>,30,0.21



age (present=13,949, missing=416, unique=6)


,count,pct
age,,
65+,3522,24.52
55-64,2500,17.40
35-44,2441,16.99
25-34,2374,16.53
45-54,2141,14.90
18-24,971,6.76
<NA>,416,2.90


... (34 outputs left out)


personal_income (present=13,502, missing=863, unique=27)


,count,pct
personal_income,,
No income,5088,35.42
<NA>,863,6.01
"$60,000-$74,999",848,5.90
"$40,000-$49,999",760,5.29
"$50,000-$59,999",718,5.00
"$75,000-$89,999",570,3.97
"$30,000-$34,999",529,3.68
"$90,000-$109,999",483,3.36
"$35,000-$39,999",462,3.22



occupation (present=13,702, missing=663, unique=322)


,count,pct
occupation,,
<NA>,663,4.62
Primary school teachers,359,2.50
Business services and administration managers not elsewhere classified,332,2.31
Nursing professionals,326,2.27
Manufacturing labourers not elsewhere classified,315,2.19
...,...,...
Enquiry clerks,1,0.01
Hunters and trappers,1,0.01
Other cleaning workers,1,0.01


# Twin2k

In [5]:
inspect_dataset_personas("twin2k")

=== twin2k: 2,058 participants | 15 active persona attributes ===

country (present=2,058, missing=0, unique=1)


,count,pct
country,,
United States,2058,100.0



gender (present=2,058, missing=0, unique=2)


,count,pct
gender,,
Female,1044,50.73
Male,1014,49.27



age (present=2,058, missing=0, unique=4)


,count,pct
age,,
30-49,735,35.71
50-64,658,31.97
18-29,388,18.85
65+,277,13.46


... (20 outputs left out)


household_size (present=2,058, missing=0, unique=5)


,count,pct
household_size,,
2,650,31.58
3,423,20.55
1,412,20.02
4,352,17.10
More than 4,221,10.74



citizen (present=2,058, missing=0, unique=2)


,count,pct
citizen,,
Yes,2054,99.81
No,4,0.19


# WVS7

In [6]:
inspect_dataset_personas("wvs7")

=== wvs7: 2,596 participants | 29 active persona attributes ===

country (present=2,596, missing=0, unique=1)


,count,pct
country,,
United States,2596,100.0



gender (present=2,596, missing=0, unique=2)


,count,pct
gender,,
Male,1390,53.54
Female,1206,46.46



age (present=2,596, missing=0, unique=6)


,count,pct
age,,
25-34,702,27.04
35-44,455,17.53
45-54,433,16.68
55-64,379,14.60
65+,340,13.10
18-24,287,11.06


... (48 outputs left out)


social_class (present=2,561, missing=35, unique=5)


,count,pct
social_class,,
Lower middle class,967,37.25
Upper middle class,762,29.35
Working class,615,23.69
Lower class,183,7.05
<NA>,35,1.35
Upper class,34,1.31



voting_frequency (present=2,562, missing=34, unique=4)


,count,pct
voting_frequency,,
Always,1582,60.94
Usually,546,21.03
Never,356,13.71
Not allowed to vote,78,3.00
<NA>,34,1.31


# Subpop

In [7]:
inspect_dataset_personas("subpop")

=== subpop: 22 participants | 9 active persona attributes ===

country (present=22, missing=0, unique=1)


,count,pct
country,,
United States,22,100.0



gender (present=2, missing=20, unique=2)


,count,pct
gender,,
<NA>,20,90.91
Male,1,4.55
Female,1,4.55



region (present=2, missing=20, unique=2)


,count,pct
region,,
<NA>,20,90.91
Northeast,1,4.55
South,1,4.55


... (8 outputs left out)


political_party (present=2, missing=20, unique=2)


,count,pct
political_party,,
<NA>,20,90.91
Democrat,1,4.55
Republican,1,4.55



political_ideology (present=3, missing=19, unique=3)


,count,pct
political_ideology,,
<NA>,19,86.36
Liberal,1,4.55
Conservative,1,4.55
Moderate,1,4.55


# Covid

In [3]:
inspect_dataset_personas("covid")

=== covid: 1,971 participants | 10 active persona attributes ===

country (present=1,971, missing=0, unique=1)


,count,pct
country,,
United States,1971,100.0



gender (present=1,971, missing=0, unique=3)


,count,pct
gender,,
Female,999,50.68
Male,949,48.15
Prefer not to say,23,1.17



age (present=1,971, missing=0, unique=6)


,count,pct
age,,
35-44,382,19.38
25-34,377,19.13
45-54,360,18.26
65+,317,16.08
18-24,274,13.90
55-64,261,13.24


... (10 outputs left out)


political_party (present=1,971, missing=0, unique=4)


,count,pct
political_party,,
Democrat,737,37.39
Republican,675,34.25
Independent,436,22.12
Other/don't know,123,6.24



political_ideology (present=1,971, missing=0, unique=7)


,count,pct
political_ideology,,
Moderate,660,33.49
Liberal,286,14.51
Very liberal,258,13.09
Very conservative,226,11.47
Conservative,211,10.71
Somewhat conservative,178,9.03
Somewhat liberal,152,7.71


# Grummon 2023

In [7]:
inspect_dataset_personas("grummon2023")

=== grummon2023: 7,945 participants | 14 active persona attributes ===

country (present=7,945, missing=0, unique=1)


,count,pct
country,,
United States,7945,100.0



gender (present=7,945, missing=0, unique=4)


,count,pct
gender,,
Female,4078,51.33
Male,3779,47.56
Non-binary,69,0.87
Other,19,0.24



age (present=7,945, missing=0, unique=6)


,count,pct
age,,
65+,1699,21.38
25-34,1365,17.18
55-64,1328,16.71
45-54,1294,16.29
35-44,1279,16.10
18-24,980,12.33


... (18 outputs left out)


medical_reading_help (present=7,945, missing=0, unique=5)


,count,pct
medical_reading_help,,
Never,4201,52.88
Rarely,1908,24.02
Sometimes,1224,15.41
Most of the time,382,4.81
Always,230,2.89



bmi (present=7,931, missing=14, unique=4)


,count,pct
bmi,,
Healthy weight (18.5 to <25.0),2720,34.24
Obese (>=30.0),2514,31.64
Overweight (25.0 to <30.0),2385,30.02
Underweight (<18.5),312,3.93
<NA>,14,0.18


# Grummon 2025

In [5]:
inspect_dataset_personas("grummon2025")

=== grummon2025: 5,636 participants | 10 active persona attributes ===

country (present=5,636, missing=0, unique=1)


,count,pct
country,,
United States,5636,100.0



gender (present=5,629, missing=7, unique=4)


,count,pct
gender,,
Female,3400,60.33
Male,2111,37.46
Non-binary,112,1.99
<NA>,7,0.12
Other,6,0.11



age (present=5,636, missing=0, unique=6)


,count,pct
age,,
25-34,1721,30.54
35-44,1660,29.45
45-54,967,17.16
55-64,565,10.02
18-24,432,7.67
65+,291,5.16


... (10 outputs left out)


children_in_household (present=5,618, missing=18, unique=8)


,count,pct
children_in_household,,
0,3569,63.33
1,905,16.06
2,816,14.48
3,253,4.49
4,56,0.99
<NA>,18,0.32
5,13,0.23
6,4,0.07
7 or more,2,0.04



nutrition_label_use (present=5,620, missing=16, unique=5)


,count,pct
nutrition_label_use,,
Sometimes,2561,45.44
Most of the time,1638,29.06
Rarely,768,13.63
Always,469,8.32
Never,184,3.26
<NA>,16,0.28


# Berke 2023

In [6]:
inspect_dataset_personas("berke2023")

=== berke2023: 3,715 participants | 8 active persona attributes ===

country (present=3,715, missing=0, unique=1)


,count,pct
country,,
United States,3715,100.0



gender (present=3,715, missing=0, unique=4)


,count,pct
gender,,
Male,1823,49.07
Female,1812,48.78
Other,68,1.83
Prefer not to say,12,0.32



age (present=3,715, missing=0, unique=6)


,count,pct
age,,
25-34,1253,33.73
35-44,897,24.15
18-24,600,16.15
45-54,457,12.30
55-64,350,9.42
65+,158,4.25


... (6 outputs left out)


urban_rural (present=3,693, missing=22, unique=3)


,count,pct
urban_rural,,
Suburban,2047,55.10
Urban,1020,27.46
Rural,626,16.85
<NA>,22,0.59



residence_type (present=3,715, missing=0, unique=3)


,count,pct
residence_type,,
Private home,2644,71.17
Apartment,978,26.33
Other,93,2.50


# Kriner 2026

In [3]:
inspect_dataset_personas("kriner2026")

=== kriner2026: 1,000 participants | 14 active persona attributes ===

country (present=1,000, missing=0, unique=1)


,count,pct
country,,
United States,1000,100.0



gender (present=1,000, missing=0, unique=3)


,count,pct
gender,,
Female,512,51.2
Male,483,48.3
Prefer not to say,5,0.5



age (present=1,000, missing=0, unique=6)


,count,pct
age,,
65+,226,22.6
25-34,196,19.6
35-44,177,17.7
45-54,159,15.9
55-64,139,13.9
18-24,103,10.3


... (18 outputs left out)


voted_last_election (present=998, missing=2, unique=4)


,count,pct
voted_last_election,,
Yes,664,66.4
No,247,24.7
Not registered to vote,77,7.7
Too young to vote,10,1.0
<NA>,2,0.2



vote_choice_2024 (present=664, missing=336, unique=3)


,count,pct
vote_choice_2024,,
<NA>,336,33.6
Kamala Harris,300,30.0
Donald Trump,283,28.3
Someone else,81,8.1


# Wu 2025

In [8]:
inspect_dataset_personas("wu2025")

=== wu2025: 1,862 participants | 19 active persona attributes ===

country (present=1,862, missing=0, unique=1)


,count,pct
country,,
United States,1862,100.0



gender (present=1,850, missing=12, unique=2)


,count,pct
gender,,
Female,973,52.26
Male,877,47.10
<NA>,12,0.64



age (present=1,862, missing=0, unique=8)


,count,pct
age,,
55-64,421,22.61
25-34,364,19.55
35-44,352,18.90
65-74,234,12.57
45-54,208,11.17
18-24,198,10.63
75-84,75,4.03
85+,10,0.54


... (28 outputs left out)


military_service (present=1,847, missing=15, unique=2)


,count,pct
military_service,,
No,1586,85.18
Yes,261,14.02
<NA>,15,0.81



electric_heating (present=1,808, missing=54, unique=2)


,count,pct
electric_heating,,
Yes,1140,61.22
No,668,35.88
<NA>,54,2.90


# Kantorowicz 2026

In [9]:
inspect_dataset_personas("kantorowicz2026")

=== kantorowicz2026: 2,242 participants | 5 active persona attributes ===

country (present=2,242, missing=0, unique=1)


,count,pct
country,,
United States,2242,100.0



gender (present=2,211, missing=31, unique=2)


,count,pct
gender,,
Female,1127,50.27
Male,1084,48.35
<NA>,31,1.38



age (present=2,222, missing=20, unique=2)


,count,pct
age,,
Older,1148,51.20
Younger,1074,47.90
<NA>,20,0.89



education (present=2,240, missing=2, unique=2)


,count,pct
education,,
Below higher education,1215,54.19
Higher education,1025,45.72
<NA>,2,0.09



income (present=2,113, missing=129, unique=2)


,count,pct
income,,
Higher income,1324,59.05
Lower income,789,35.19
<NA>,129,5.75


# Magistro 2025

In [10]:
inspect_dataset_personas("magistro2025")

=== magistro2025: 6,056 participants | 8 active persona attributes ===

country (present=6,056, missing=0, unique=2)


,count,pct
country,,
Canada,3031,50.05
United States,3025,49.95



gender (present=6,055, missing=1, unique=3)


,count,pct
gender,,
Female,3277,54.11
Male,2699,44.57
Other,79,1.30
<NA>,1,0.02



age (present=6,056, missing=0, unique=4)


,count,pct
age,,
45-64,2269,37.47
30-44,1383,22.84
65+,1382,22.82
18-29,1022,16.88


... (6 outputs left out)


income (present=5,271, missing=785, unique=3)


,count,pct
income,,
Middle,2996,49.47
High,1173,19.37
Low,1102,18.20
<NA>,785,12.96



political_party (present=6,056, missing=0, unique=6)


,count,pct
political_party,,
Other,2249,37.14
Democrat,1056,17.44
Conservative Party of Canada,786,12.98
Republican,755,12.47
Liberal Party of Canada,738,12.19
New Democratic Party,472,7.79
